*** trend scanning notes ***

In [ ]:
import numpy as np
import statsmodels.api as sml
import pandas as pd 

def tValLinR(close):
    # t value from a linear trend 

    x = np.ones((close.shape[0],2)) 
    # Create a matrix "x" with shape (N,2)
    # Column 0 = all ones (Acts as the intercept term)
    # Column 1 = will store time index(trend component)

    x[:,1] = np.arange(close.shape[0])
    # Fill second column with sequential time index: 0, 1, 2, ... n-1
    
    ols = sml.OLS(close,x).fit()
    # close = a + b * time_index 

    return ols.tvalues[1]
    # Return the t-stats for the slope coefficient (trend strength)
    # index [1] refers to the time trend coefficient 

    



Next step, we can try a set of alternative values for L, and pick the value of L that maximizes t hat. In this way, we label x according to the most statistically significant trend observed in the future, out of multiple possible look-forward periods. The argument are:

molecule (the index of observations we wish to label), 
close (the time series of {x_t}),
span (the set of values of L that the algorithm will evaluate, in search for the maximum absolute t-value),

Output is a data frame where:
the index is the timestamp of the x_t, 
column t1 reports the timestamp of the farthest observation used to find the most significant trend. 
column tVal reports the t-value associated with the most significant linear trend among the set of valuated look-forward periods, and column bin is the label (y_t)


In [ ]:
def getBinsFromTrend(molecule, close, span):
    '''
    Derive labels from the sign of t-value of linear trend
    Output includes:
    - t1: End time for the identified trend
    - tVal: t-value associated with the estimated trend coefficient 
    - bin: Sign of the trend
    '''

    out = pd.DataFrame(index = molecule, columns = ['t1', 'tVal','bin']) 
    # Prepare the result table 

    hrzns = xrange(*span)
    # Create a list of future windows like [5,10,15,20,...,50]

    for dt0 in molecule: 

        df0 = pd.Series()
        # Create the container

        iloc0 = close.index.get_loc(dt0)
        # Converts a label(data) into a position
        # eg: turn 2020-01-02 to 2 
        # get_loc: find the integer position of this label in the index, following the previous code

        if iloc0 + max(hrzns) > close.shape[0]:
        # If we dont have enough future data -> skip
            continue 

        for hrzn in hrzns:
            dt1 = close.index[iloc0 + hrzn - 1]
        # Find future endpoint (start time + horizon)

            df1 = close.loc[dt0:dt1] 
            # From start to end of this horizon

            df0.loc[dt1] = tValLinR(df1.values)
            # Run regression using price ~ time, then store the t-stats for this horizon 

            # At this point, for each horizon, we now have: 
            # (5,1.2), (10,2.5), (20,-0.8) ... 
            # So we pick the best one: 

        dt1 = df0.replace([-np.inf, np.inf, np.nan],0).abs().idmax()
        # Remove bad values, take absolute value, pick largest t-value 

        out.loc[dt0,['t1','tval','bin']] = df0.index[-1],df0[dt1],np.sign(df0[dt1]) 
        # prevent leakage 

    return out.dropna(subset = ['bin'])






    